EASY 1: Precision Agriculture Field Inspection

In [4]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import to_date,year,month,avg


def load_field_inspections(spark: SparkSession, path: str) -> DataFrame:
    df=spark.read.csv(path,header=True,inferSchema=True)
    return df.withColumn("inspection_date",to_date("inspection_date"))

def drop_incomplete_inspections(df: DataFrame) -> DataFrame:
    df=df.dropna(subset=["inspection_id","field_id","moisture_level"])
    return df

def fill_missing_inspector(df: DataFrame) -> DataFrame:
    return df.fillna({"inspector_name":"Unknown"})

def add_inspection_year_month(df: DataFrame) -> DataFrame:
    df=(df.withColumn("inspection_year",year("inspection_date"))
       .withColumn("inspection_month",month("inspection_date"))
    )
    return df

def average_moisture_by_crop(df: DataFrame) -> DataFrame:
    df=df.groupby("crop_type").agg(avg("moisture_level").alias("avg_moisture"))
    return df.select("crop_type","avg_moisture")

# =========================
# MAIN CODE
# =========================

spark = SparkSession.builder \
    .appName("Field Inspections") \
    .getOrCreate()


# Dataset path
path = "/content/field_inspections.csv"


# Function 1
df = load_field_inspections(
    spark,
    path
)
print("load_field_inspections output:")
df.show()


# Function 2
df = drop_incomplete_inspections(
    df
)
print("drop_incomplete_inspections output:")
df.show()


# Function 3
df = fill_missing_inspector(
    df
)
print("fill_missing_inspector output:")
df.show()


# Function 4
df = add_inspection_year_month(
    df
)
print("add_inspection_year_month output:")
df.show()


# Function 5
result = average_moisture_by_crop(
    df
)
print("average_moisture_by_crop output:")
result.show()



# Optional: stop Spark
spark.stop()

load_field_inspections output:
+-------------+--------+---------+---------------+--------------+--------------+
|inspection_id|field_id|crop_type|inspection_date|moisture_level|inspector_name|
+-------------+--------+---------+---------------+--------------+--------------+
|         I001|    F101|     Rice|     2026-06-05|          72.5|         Meena|
|         I002|    F102|    Wheat|     2026-06-07|          48.0|          NULL|
|         I003|    F103|   Cotton|     2026-07-01|          39.5|          Ravi|
|         I004|    F104|     Rice|     2026-07-03|          68.5|         Anita|
|         I005|    NULL|    Wheat|     2026-07-05|          51.0|        Suresh|
|         I006|    F106|   Cotton|     2026-08-10|          NULL|         Meena|
|         I007|    F107|     Rice|     2026-08-15|          75.0|          NULL|
|         I008|    F108|    Wheat|     2026-08-20|          49.0|         Anita|
|         I009|    F109|   Cotton|     2026-09-02|          41.5|          Rav

EASY 2: Museum Artifact Catalog
Insights

In [8]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    col,
    to_date,
    year,
    month,
    sum
)


# Function 1 — Load Artifact Data
def load_artifact_data(spark: SparkSession, path: str) -> DataFrame:
    df = spark.read.csv(path, header=True, inferSchema=True)
    df = df.withColumn("acquisition_date", to_date(col("acquisition_date")))
    return df


# Function 2 — Drop Records Without Acquisition Date
def drop_records_without_acquisition_date(df: DataFrame) -> DataFrame: return df.dropna(subset=["artifact_id", "acquisition_date"])


# Function 3 — Fill Missing Category
def fill_missing_category(df: DataFrame) -> DataFrame: return df.fillna("Unclassified", subset=["category"])


# Function 4 — Add Acquisition Year and Month
def add_acquisition_year_month(df: DataFrame) -> DataFrame:
    return df.withColumn("acquisition_year", year(col("acquisition_date"))).withColumn("acquisition_month", month(col("acquisition_date")))


# Function 5 — Highest Value Gallery
def highest_value_gallery(df: DataFrame) -> tuple:
    result = (
        df.groupBy("gallery")
          .agg(sum("estimated_value").alias("total_value"))
          .orderBy(col("total_value").desc(), col("gallery").asc())
          .first()
    )
    if result is None: return ("", 0.0)
    return (result["gallery"], float(result["total_value"]))

spark = SparkSession.builder \
    .appName("Museum Artifact Catalog") \
    .getOrCreate()

path = "/content/artifacts.csv"

print("load_artifact_data output:")
df = load_artifact_data(spark, path)
df.show()

print("drop_records_without_acquisition_date output:")
df = drop_records_without_acquisition_date(df)
df.show()

print("fill_missing_category output:")
df = fill_missing_category(df)
df.show()

print("add_acquisition_year_month output:")
df = add_acquisition_year_month(df)
df.show()

print("highest_value_gallery output:")
df = highest_value_gallery(df)
print(df)


load_artifact_data output:
+-----------+-----------------+-----------+----------------+---------------+---------+
|artifact_id|    artifact_name|   category|acquisition_date|estimated_value|  gallery|
+-----------+-----------------+-----------+----------------+---------------+---------+
|       A001|      Bronze Lamp|  Metalwork|      2024-01-10|         120000|Gallery A|
|       A002|Temple Manuscript|       NULL|      2024-02-05|         250000|Gallery B|
|       A003|       Stone Idol|  Sculpture|      2024-03-15|         180000|Gallery A|
|       A004|       Royal Coin|Numismatics|      2025-01-22|          95000|Gallery C|
|       A005|  Ancient Textile|    Textile|            NULL|          70000|Gallery B|
|       A006|      Wooden Mask|       NULL|      2025-02-14|          85000|Gallery C|
|       A007| Palm Leaf Record| Manuscript|      2025-03-10|         130000|Gallery B|
|       A008| Ceremonial Sword|  Metalwork|      2025-03-18|         160000|Gallery A|
+-----------+---

MEDIUM 1:Aircraft Maintenance
Compliance Analytics

In [11]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import *
from pyspark.sql.window import Window


def define_maintenance_schema() -> StructType:
    return StructType([
        StructField("maintenance_id",StringType(),True),
        StructField("aircraft_id",StringType(),True),
        StructField("maintenance_date",StringType(),True),
        StructField("maintenance_cost",DoubleType(),True),
        StructField("maintenance_status",StringType(),True),
    ])


def load_maintenance_data(spark: SparkSession, path: str, schema: StructType) -> DataFrame:
    df=spark.read.csv(path,header=True,schema=schema)
    return df.withColumn("maintenance_date",to_date("maintenance_date"))

def join_aircraft_with_maintenance(aircraft_df: DataFrame, maintenance_df: DataFrame) -> DataFrame:
    return aircraft_df.join(maintenance_df,"aircraft_id","inner")

def aircraft_without_maintenance(aircraft_df: DataFrame, maintenance_df: DataFrame) -> DataFrame:
    return aircraft_df.join(maintenance_df,"aircraft_id","left_anti")

def rank_aircraft_by_maintenance_cost(df: DataFrame) -> DataFrame:
    df=df.groupby(["aircraft_model","aircraft_id"]).agg(sum("maintenance_cost").alias("total_maintenance_cost"))
    window_spec=Window.partitionBy("aircraft_model").orderBy(col("total_maintenance_cost").desc())
    return df.withColumn("maintenance_rank",rank().over(window_spec))

# =========================
# MAIN CODE
# =========================

spark = SparkSession.builder \
    .appName("Aircraft Maintenance Analysis") \
    .getOrCreate()

# Dataset paths
aircraft_path = "/content/aircraft.csv"
maintenance_path = "/content/maintenance.csv"

# Define schema for maintenance data
maintenance_schema = define_maintenance_schema()

# Load aircraft data (assuming header and infer schema)
print("Loading aircraft data:")
aircraft_df = spark.read.csv(aircraft_path, header=True, inferSchema=True)
aircraft_df.show()

# Function 1: Load Maintenance Data
print("load_maintenance_data output:")
maintenance_df = load_maintenance_data(spark, maintenance_path, maintenance_schema)
maintenance_df.show()

# Function 2: Join Aircraft with Maintenance
print("join_aircraft_with_maintenance output:")
joined_df = join_aircraft_with_maintenance(aircraft_df, maintenance_df)
joined_df.show()

# Function 3: Aircraft without Maintenance
print("aircraft_without_maintenance output:")
no_maintenance_df = aircraft_without_maintenance(aircraft_df, maintenance_df)
no_maintenance_df.show()

# Function 4: Rank Aircraft by Maintenance Cost
print("rank_aircraft_by_maintenance_cost output:")
ranked_df = rank_aircraft_by_maintenance_cost(joined_df)
ranked_df.show()

# Optional: stop Spark
spark.stop()

Loading aircraft data:
+-----------+--------------+------------+--------------+
|aircraft_id|aircraft_model|base_airport|service_status|
+-----------+--------------+------------+--------------+
|      AC101|          A320|     Chennai|        Active|
|      AC102|          A320|       Delhi|        Active|
|      AC103|          B737|      Mumbai|        Active|
|      AC104|          B737|   Hyderabad|      Grounded|
|      AC105|          A321|   Bengaluru|        Active|
|      AC106|          A321|       Kochi|        Active|
+-----------+--------------+------------+--------------+

load_maintenance_data output:
+--------------+-----------+----------------+----------------+------------------+
|maintenance_id|aircraft_id|maintenance_date|maintenance_cost|maintenance_status|
+--------------+-----------+----------------+----------------+------------------+
|          M001|      AC101|      2026-05-01|         45000.0|         Completed|
|          M002|      AC101|      2026-06-10|   

MEDIUM 2:Film Production Crew
Payment Analytics

In [13]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)
from pyspark.sql.functions import (
    col,
    to_date,
    year,
    month,
    sum,
    rank
)
from pyspark.sql.window import Window


# Function 1 — Define Payment Schema
def define_payment_schema() -> StructType: return StructType([
    StructField("payment_id", StringType(), True),
    StructField("crew_id", StringType(), True),
    StructField("payment_date", StringType(), True),
    StructField("payment_amount", DoubleType(), True),
    StructField("payment_status", StringType(), True)
])


# Function 2 — Load and Clean Payments
def load_and_clean_payments(spark: SparkSession, path: str, schema: StructType) -> DataFrame:
    df = spark.read.csv(path, header=True, schema=schema)
    df = df.withColumn("payment_date", to_date(col("payment_date")))
    df = df.dropna(subset=["payment_id", "crew_id", "payment_amount"])
    df = df.fillna("Pending", subset=["payment_status"])
    df = df.withColumn("payment_year", year(col("payment_date"))).withColumn("payment_month", month(col("payment_date")))
    return df


# Function 3 — Join Crew With Payments
def join_crew_with_payments(crew_df: DataFrame, payments_df: DataFrame) -> DataFrame: return crew_df.join(payments_df, on="crew_id", how="inner")


# Function 4 — Crew Without Payments
def crew_without_payments(crew_df: DataFrame, payments_df: DataFrame) -> DataFrame: return crew_df.join(payments_df, on="crew_id", how="left_anti")


# Function 5 — Rank Crew by Total Payment
def rank_crew_by_total_payment(df: DataFrame) -> DataFrame:
    result = df.groupBy("department", "crew_id", "crew_name").agg(sum("payment_amount").alias("total_payment"))
    window_spec = Window.partitionBy("department").orderBy(col("total_payment").desc())
    result = result.withColumn("payment_rank", rank().over(window_spec))
    return result.select("department", "crew_id", "crew_name", "total_payment", "payment_rank")

# =========================
# MAIN CODE
# =========================

spark = SparkSession.builder \
    .appName("Crew Payment Analysis") \
    .getOrCreate()

# Dataset paths
crew_path = "/content/crew.csv"
payments_path = "/content/payments.csv"

# Define schema for payments data
payment_schema = define_payment_schema()

# Load and clean payments data
print("load_and_clean_payments output:")
payments_df = load_and_clean_payments(spark, payments_path, payment_schema)
payments_df.show()

# Load crew data
print("Loading crew data:")
crew_df = spark.read.csv(crew_path, header=True, inferSchema=True)
crew_df.show()

# Function 3: Join Crew With Payments
print("join_crew_with_payments output:")
joined_crew_payments_df = join_crew_with_payments(crew_df, payments_df)
joined_crew_payments_df.show()

# Function 4: Crew Without Payments
print("crew_without_payments output:")
crew_no_payments_df = crew_without_payments(crew_df, payments_df)
crew_no_payments_df.show()

# Function 5: Rank Crew by Total Payment
print("rank_crew_by_total_payment output:")
ranked_crew_df = rank_crew_by_total_payment(joined_crew_payments_df)
ranked_crew_df.show()

# Optional: stop Spark
spark.stop()

load_and_clean_payments output:
+----------+-------+------------+--------------+--------------+------------+-------------+
|payment_id|crew_id|payment_date|payment_amount|payment_status|payment_year|payment_month|
+----------+-------+------------+--------------+--------------+------------+-------------+
|      P001|   C101|  2026-01-15|       75000.0|          Paid|        2026|            1|
|      P002|   C101|  2026-02-15|       80000.0|          Paid|        2026|            2|
|      P003|   C102|  2026-01-20|       70000.0|       Pending|        2026|            1|
|      P004|   C103|  2026-02-10|       65000.0|          Paid|        2026|            2|
|      P005|   C104|  2026-02-18|       65000.0|          Paid|        2026|            2|
|      P006|   C105|  2026-03-01|       90000.0|          Paid|        2026|            3|
|      P007|   C105|  2026-04-01|       90000.0|          Paid|        2026|            4|
|      P008|   C106|  2026-03-05|       85000.0|          